In [ ]:
# Install deps
!pip install -q pandas scikit-learn joblib tqdm datasets

In [ ]:
import os, zipfile, requests, sys
url = "https://github.com/vedangvatsa/ai-detection-at-scale/archive/refs/heads/main.zip"
r = requests.get(url)
open("repo.zip", "wb").write(r.content)
with zipfile.ZipFile("repo.zip") as z:
    z.extractall(".")
os.rename("ai-detection-at-scale-main", "ai-detection-at-scale")
os.chdir("ai-detection-at-scale")
sys.path.insert(0, '.')
print("Repo ready")

In [ ]:
# Download model assets
!python scripts/download_assets.py

In [ ]:
# TuringBench Evaluation
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
from tool.feature_extractor import extract_features
import joblib

# Load stylometric model
model_data = joblib.load("models/detector_all.joblib")
model = model_data["model"] if isinstance(model_data, dict) else model_data

# Load TuringBench from HuggingFace (with fallback)
from datasets import load_dataset
tb = None
test_df = None
try:
    tb = load_dataset("turingbench/TuringBench", revision="refs/convert/parquet")
except Exception as e:
    print(f"Primary load failed: {e}")
    print("Trying alternative mirror...")
    try:
        tb = load_dataset("liam-hp/TuringBench")
    except Exception as e2:
        print(f"Mirror also failed: {e2}")
        print("Falling back to HC3 for binary detection benchmark...")
        hc3 = load_dataset("Hello-SimpleAI/HC3", name="default", revision="refs/convert/parquet")
        rows = []
        for item in hc3['train']:
            for ans in item.get('human_answers', []):
                if isinstance(ans, str) and len(ans.strip()) > 20:
                    rows.append({'text': ans, 'label': 'human'})
            for ans in item.get('chatgpt_answers', []):
                if isinstance(ans, str) and len(ans.strip()) > 20:
                    rows.append({'text': ans, 'label': 'chatgpt'})
        import pandas as pd
        test_df = pd.DataFrame(rows)
        tb = None

if tb is not None:
    print(f"Columns: {tb['train'].column_names}")
    print(f"Labels: {set(tb['train']['label'][:100])}")
    test_df = tb['test'].to_pandas() if 'test' in tb else tb['train'].to_pandas()

sample_size = min(5000, len(test_df))
test_sample = test_df.sample(n=sample_size, random_state=42)
print(f"Evaluating on {len(test_sample)} texts")

# Extract features and predict
features = []
labels = []
for i, row in test_sample.iterrows():
    text = row['text'] if 'text' in row else row.get('generation', row.get('Generation', ''))
    label_raw = row['label']
    # Binary: human=0, any AI model=1
    if isinstance(label_raw, str):
        label = 0 if label_raw.lower() in ('human', '0') else 1
    else:
        label = 0 if int(label_raw) == 0 else 1
    
    if not isinstance(text, str) or len(text.strip()) < 20:
        continue
    
    feats = extract_features(text, extended=False)
    if feats is None:
        continue
    
    X = np.array([feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                                        'connector_density', 'hedge_density', 'mean_sent_len',
                                        'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']])
    features.append(X)
    labels.append(label)

features = np.array(features)
labels = np.array(labels)

# Predict
probs = model.predict_proba(features)[:, 1]
preds = (probs >= 0.5).astype(int)
auc = roc_auc_score(labels, probs)
acc = accuracy_score(labels, preds)
print(f"TuringBench AUC: {auc:.4f}")
print(f"TuringBench Accuracy: {acc:.4f}")

# Save results
results = pd.DataFrame([{'benchmark': 'TuringBench', 'auc': auc, 'accuracy': acc, 'n_samples': len(labels)}])
results.to_csv('/kaggle/working/turingbench_results.csv', index=False)
print("Results saved")
